## Part 1

In [393]:
# Uncomment these if running for the first time

# !pip install playwright pandas
# !playwright install chromium

In [394]:
from playwright.async_api import async_playwright
import pandas as pd
import time
import re

In [395]:
URL = "https://www.booking.com/searchresults.en-gb.html?ss=Islamabad&ssne=Islamabad&ssne_untouched=Islamabad&efdco=1&label=en-pk-booking-desktop-732gBu1H4WlF1HDSvlYKKAS652796017653%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-334108349%3Alp1011080%3Ali%3Adec%3Adm&sid=3f6cb1e41e124ed987c7ba78907848c9&aid=2311236&lang=en-gb&sb=1&src_elem=sb&src=index&dest_id=-2762812&dest_type=city&checkin=2026-08-06&checkout=2026-08-07&ltfd=5%3A1%3A8-2026_9-2026_10-2026%3A1%3A&group_adults=2&no_rooms=1&group_children=0"

HEADLESS = False

SLOW_MO = 0

In [396]:
playwright = await async_playwright().start()

browser = await playwright.chromium.launch(
    headless=HEADLESS,
    slow_mo=SLOW_MO
)

context = await browser.new_context(
    viewport={"width": 1600, "height": 900},
    locale="en-US",
    java_script_enabled=True
)

page = await context.new_page()

await page.goto(URL)

await page.wait_for_load_state("networkidle")

print(await page.title())

Booking.com: Hotels in Islamabad. Book your hotel now!


In [397]:
# time.sleep(10)

In [398]:
try:
    dismiss_btn = page.locator('button[aria-label="Dismiss sign in information."]')
    await dismiss_btn.wait_for(state="visible", timeout=5000)
    await dismiss_btn.click()
    print("Sign-in discount popup closed")
except:
    print("Sign-in discount popup did not appear")

Sign-in discount popup closed


In [399]:
# time.sleep(5)

In [400]:
possible_buttons = [
    "Accept",
    "Accept all",
    "I agree",
    "Got it",
    "Allow all"
]

for text in possible_buttons:
    try:
        btn = page.get_by_role("button", name=text)

        if await btn.count() > 0:
            await btn.first.click(timeout=2000)
            print("Cookie popup closed")
            break

    except:
        pass

In [401]:
cards = page.locator('[data-testid="property-card"]')

print("Properties currently loaded:", await cards.count())

Properties currently loaded: 25


In [402]:
for i in range(3):
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
    await page.wait_for_timeout(1000)
    print(f"Scrolled to bottom ({i + 1}/3)")

Scrolled to bottom (1/3)
Scrolled to bottom (2/3)
Scrolled to bottom (3/3)


## Part 2

In [403]:
from playwright.async_api import Error as PlaywrightError

async def load_all_results(page):

    previous_count = 0

    while True:

        try:

            cards = page.locator('[data-testid="property-card"]')

            current_count = await cards.count()

        except PlaywrightError as e:

            print(f"Browser/page closed unexpectedly ({e}). Stopping.")

            break

        print(f"Loaded properties : {current_count}")

        if current_count == previous_count:
            print("No new properties detected.")

        previous_count = current_count

        load_more = page.get_by_role(
            "button",
            name=re.compile("Load more", re.IGNORECASE)
        )

        if await load_more.count() == 0:
            print("No Load More button found.")
            break

        try:

            await load_more.first.scroll_into_view_if_needed()

            await page.wait_for_timeout(400)

            await load_more.first.click(timeout=5000)

            print("Clicked Load More")

        except PlaywrightError as e:

            print(f"Browser/page closed unexpectedly ({e}). Stopping.")

            break

        except Exception as e:

            print(f"Could not click button: {e}")

            break

        try:

            await page.wait_for_function(
                f"""
                () => document.querySelectorAll(
                '[data-testid="property-card"]'
                ).length > {current_count}
                """,
                timeout=20000
            )

        except PlaywrightError as e:

            print(f"Browser/page closed unexpectedly ({e}). Stopping.")

            break

        except:

            print("No additional properties loaded.")

            break

        await page.wait_for_timeout(400)

    print("Finished Loading")

In [404]:
await load_all_results(page)

Loaded properties : 75
Clicked Load More
Loaded properties : 99
Clicked Load More
Loaded properties : 123
Clicked Load More
Loaded properties : 147
Clicked Load More
Loaded properties : 172
Clicked Load More
Loaded properties : 195
Clicked Load More
Loaded properties : 220
Clicked Load More
Loaded properties : 244
Clicked Load More
Loaded properties : 269
Clicked Load More
Loaded properties : 291
Clicked Load More
Loaded properties : 316
Clicked Load More
Loaded properties : 338
Clicked Load More
Loaded properties : 361
Clicked Load More
Loaded properties : 386
Clicked Load More
Loaded properties : 411
Clicked Load More
Loaded properties : 435
Clicked Load More
Loaded properties : 459
Clicked Load More
Loaded properties : 483
Clicked Load More
Loaded properties : 508
Clicked Load More
Loaded properties : 532
Clicked Load More
Loaded properties : 553
Clicked Load More
Loaded properties : 576
Clicked Load More
Loaded properties : 600
Clicked Load More
Loaded properties : 625
Clicked Load

In [405]:
cards = page.locator('[data-testid="property-card"]')

TOTAL_PROPERTIES = await cards.count()

print("="*50)

print("Total Properties Loaded :", TOTAL_PROPERTIES)

print("="*50)

Total Properties Loaded : 961


## PART 3

In [406]:
properties = await page.eval_on_selector_all(
    '[data-testid="property-card"]',
    """
    (cards) => {

        function text(root, selector) {
            const el = root.querySelector(selector);
            return el ? el.innerText.trim() : null;
        }

        function attr(root, selector, name) {
            const el = root.querySelector(selector);
            return el ? el.getAttribute(name) : null;
        }

        function findByText(root, regex) {
            const walker = document.createTreeWalker(root, NodeFilter.SHOW_ELEMENT);
            let node;
            while (node = walker.nextNode()) {
                if (node.children.length === 0 && regex.test(node.innerText || "")) {
                    return node.innerText.trim();
                }
            }
            return null;
        }

        function bedType(root) {
            const units = root.querySelector('[data-testid="recommended-units"]');
            return units ? findByText(units, /\\bbeds?\\b/i) : null;
        }

        return cards.map((card) => ({
            "property_name": text(card, '[data-testid="title"]'),
            "property_url": attr(card, "a", "href"),
            "Price_pkr": text(card, '[data-testid="price-and-discounted-price"]'),
            "review_score": text(card, '[data-testid="review-score"] > :nth-child(2)'),
            "review_count": text(card, '[data-testid="review-score"] > :nth-child(3) > :nth-child(2)'),
            "address": text(card, '[data-testid="address-link"]'),
            "distance_from_center_km": text(card, '[data-testid="distance"]'),
            "property_type": text(card, '[data-testid="recommended-units"] > div > div > h4'),
            "bed_type": bedType(card),
            "breakfast_included": findByText(card, /Breakfast/i),
            "free_cancellation": findByText(card, /Free cancellation/i),
            "reserve_without_payment": findByText(card, /No prepayment/i),
            "image": attr(card, "img", "src"),
            "stars": card.querySelectorAll("svg").length,
        }));
    }
    """
)

print(f"Extraction Finished ({len(properties)} properties collected)")

Extraction Finished (961 properties collected)


In [407]:
df = pd.DataFrame(properties)

## PART 4  

In [408]:
df = df.drop_duplicates(subset=["property_url"])

print(len(df))

961


In [409]:
df["Price_pkr"] = pd.to_numeric(
    df["Price_pkr"].astype(str).str.replace(r"[^\d]", "", regex=True),
    errors="coerce"
)

df["review_count"] = pd.to_numeric(
    df["review_count"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce"
)

df["distance_from_center_km"] = pd.to_numeric(
    df["distance_from_center_km"].astype(str).str.extract(r"([\d.]+)")[0],
    errors="coerce"
)

df.head()

,property_name,property_url,Price_pkr,review_score,review_count,address,distance_from_center_km,property_type,bed_type,breakfast_included,free_cancellation,reserve_without_payment,image,stars
0,Islamabad Serena Hotel,https://www.booking.com/hotel/pk/islamabad-ser...,47240,8.7,976.0,Islamabad,1.8,Deluxe Double Room,1 extra-large double bed,Breakfast included,Free cancellation,No prepayment needed,https://cf.bstatic.com/xdata/images/hotel/squa...,14
1,Movenpick Hotel Centaurus Islamabad,https://www.booking.com/hotel/pk/movenpick-cen...,31123,8.7,670.0,"Blue Area, Islamabad",4.5,Classic King Room,1 extra-large double bed,NaN,NaN,NaN,https://cf.bstatic.com/xdata/images/hotel/squa...,14
2,Islamabad Marriott Hotel,https://www.booking.com/hotel/pk/islamabad-mar...,35700,8.5,554.0,Islamabad,0.7,"Deluxe, Guest room, 1 King",1 extra-large double bed,Breakfast included,NaN,NaN,https://cf.bstatic.com/xdata/images/hotel/squa...,14
3,Alpine Lodges,https://www.booking.com/hotel/pk/stay-well-gue...,6039,7.4,98.0,"E-11 Sector, Islamabad",12.1,Deluxe Double Room (2 Adults + 1 Child),"2 beds (1 extra-large double, 1 futon)",NaN,Free cancellation,No prepayment needed,https://cf.bstatic.com/xdata/images/hotel/squa...,11
4,Premium category Apartments,https://www.booking.com/hotel/pk/sky-line-2-in...,5950,6.4,22.0,"E-11 Sector, Islamabad",10.8,Double Room with Balcony (2 Adults + 1 Child),"2 beds (1 large double, 1 futon)",NaN,Free cancellation,No prepayment needed,https://cf.bstatic.com/xdata/images/hotel/squa...,15


In [410]:
df["breakfast_included"] = df["breakfast_included"].notna().astype(int)
df["free_cancellation"] = df["free_cancellation"].notna().astype(int)
df["reserve_without_payment"] = df["reserve_without_payment"].notna().astype(int)

In [ ]:
from pathlib import Path

output_dir = Path("../data/raw")

output_dir.mkdir(exist_ok=True, parents=True)

In [412]:
existing = list(output_dir.glob("extracted_*.csv"))

numbers = []

for file in existing:

    match = re.search(r"extracted_(\d+)\.csv", file.name)

    if match:

        numbers.append(int(match.group(1)))

next_number = max(numbers, default=0) + 1

filename = output_dir / f"extracted_{next_number}.csv"

print(filename)

Extracted Data/extracted_1.csv


In [413]:
df.to_csv(
    filename,
    index=False,
    encoding="utf-8-sig"
)

print("="*60)
print("Saved Successfully")
print(filename)
print("="*60)

Saved Successfully
Extracted Data/extracted_1.csv


In [414]:
await browser.close()

await playwright.stop()

print("Browser closed")

Browser closed
